# Lilylet NotaGen — autoregressive decoding test

Loads a trained `LilyletNotaGen` checkpoint and runs hierarchical (patch/char) autoregressive generation, mirroring NotaGen's `inference/inference.py` but adapted to the Lilylet tokenizer (256-id vocab) and deep-starry model.

Pipeline per generated patch:
1. Patch-level decoder encodes the patch sequence so far -> last hidden state.
2. Char-level decoder autoregressively samples the `patch_size` token ids inside the next patch, seeded by that hidden state.
3. The new patch is appended to the context; loop until an EOS patch `[bos, eos, ...]` appears or a length cap is hit.

Token ids are decoded back to text via the tokenizer's `text_by_id` table (NotaGen used raw `chr()`; Lilylet has protected multi-char tokens and ids > 127).

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'tests' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
import torch.nn.functional as F
from starry.utils.config import Configuration
from starry.utils.model_factory import loadModel
from starry.lilylet.data.patchifier import LilyletTokenizer

# A trained checkpoint copied to a stable temp path (best.chkpt on the training host is
# overwritten every epoch, so we work on a snapshot).
CKPT = '/tmp/lilylet-infer/best.chkpt'
CONFIG = str(REPO_ROOT / 'configs' / 'lilylet-notagenx-large.yaml')
TOKENIZER = str(REPO_ROOT / 'assets' / 'manual-tokenizer.json')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('repo:', REPO_ROOT)
print('device:', device)
print('checkpoint:', CKPT)

repo: /home/camus/work/deep-starry
device: cuda
checkpoint: /tmp/lilylet-infer/best.chkpt


In [2]:
# Build the inference model (deducer) from the training config and load weights.
config = Configuration.createOrLoad(CONFIG, volatile=True)
model = loadModel(config['model'])  # LilyletNotaGen (no Loss wrapper)

checkpoint = torch.load(CKPT, map_location='cpu')
print('checkpoint epoch:', checkpoint.get('epoch'))
missing, unexpected = model.load_state_dict(checkpoint['model'], strict=False)
print('missing keys:', len(missing), '| unexpected keys:', len(unexpected))
if missing:
    print('  e.g. missing:', missing[:3])
if unexpected:
    print('  e.g. unexpected:', unexpected[:3])

model = model.to(device).eval()
n_params = sum(p.numel() for p in model.parameters())
print('params: %.2fM' % (n_params / 1e6))
print('patch_size:', model.patch_size, '| char_vocab_size:', model.char_vocab_size)
print('pad/bos/eos:', model.special_token_id, model.bos_token_id, model.eos_token_id)

/home/camus/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


checkpoint epoch: 55
missing keys: 0 | unexpected keys: 0


params: 518.52M
patch_size: 16 | char_vocab_size: 256
pad/bos/eos: 0 1 2


In [3]:
# Tokenizer: encode prompt text into patches, decode generated ids back to text.
tokenizer = LilyletTokenizer(TOKENIZER)
PATCH_SIZE = model.patch_size
PAD, BOS, EOS = model.special_token_id, model.bos_token_id, model.eos_token_id

def patch_to_text(patch):
    """Decode one patch (list of ids) to text, mirroring NotaGen patch2chars:
    skip pad/bos, stop at eos, otherwise map id -> token text."""
    out = []
    for tid in patch:
        tid = int(tid)
        if tid == EOS:
            break
        if tid in (PAD, BOS):
            continue
        out.append(tokenizer.text_by_id.get(tid, ''))
    return ''.join(out)

def patches_to_text(patches):
    return ''.join(patch_to_text(p) for p in patches)

# sanity: round-trip a known protected token + plain chars
demo = tokenizer.encode('[r:0/3]')
print('encode "[r:0/3]" ->', demo)
print('decode back        ->', repr(patch_to_text(demo)))

encode "[r:0/3]" -> [91, 114, 58, 48, 47, 51, 93]
decode back        -> '[r:0/3]'


In [4]:
# --- sampling helpers (top-k / top-p / temperature), operating on a logits vector ---
def sample_next(logits, temperature=1.0, top_k=0, top_p=1.0):
    logits = logits.float()
    if temperature != 1.0:
        logits = logits / max(temperature, 1e-6)
    if top_k and top_k > 0:
        k = min(top_k, logits.size(-1))
        kth = torch.topk(logits, k).values[..., -1, None]
        logits = logits.masked_fill(logits < kth, float('-inf'))
    if top_p and top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True)
        probs = F.softmax(sorted_logits, dim=-1)
        cdf = probs.cumsum(dim=-1)
        remove = cdf > top_p
        remove[..., 1:] = remove[..., :-1].clone()
        remove[..., 0] = False
        sorted_logits = sorted_logits.masked_fill(remove, float('-inf'))
        logits = torch.full_like(logits, float('-inf')).scatter(-1, sorted_idx, sorted_logits)
    probs = F.softmax(logits, dim=-1)
    return int(torch.multinomial(probs, num_samples=1).item())

In [5]:
# --- char-level generation of ONE patch, conditioned on a patch hidden state ---
@torch.no_grad()
def generate_patch(encoded_patch, prefix_ids=None, temperature=1.0, top_k=0, top_p=1.0):
    """Sample the token ids inside the next patch.
    encoded_patch: [hidden_size] hidden state from the patch-level decoder.
    prefix_ids: optional ids already fixed at the start of this patch (e.g. a '[r:0/' stream marker).
    Returns a list of exactly PATCH_SIZE ids."""
    char = model.char_level_decoder
    wte = char.base.transformer.wte.weight
    # position 0 is the encoded patch state; positions 1.. are embedded char tokens.
    tokens = [BOS] + list(prefix_ids or [])
    generated = list(prefix_ids or [])
    encoded = encoded_patch.reshape(1, 1, -1)
    while len(generated) < PATCH_SIZE:
        tok_tensor = torch.tensor([tokens], device=device)
        emb = F.embedding(tok_tensor, wte)
        emb = torch.cat((encoded, emb[:, 1:, :]), dim=1)  # replace pos 0 with patch state
        logits = char.base(inputs_embeds=emb).logits[0, -1]
        nxt = sample_next(logits, temperature=temperature, top_k=top_k, top_p=top_p)
        generated.append(nxt)
        tokens.append(nxt)
    return generated

In [6]:
# --- full document generation: patch-level loop driving the char-level decoder ---
@torch.no_grad()
def generate(prompt_text='', max_patches=256, temperature=1.0, top_k=0, top_p=0.9,
             patch_stream=True, verbose=True):
    """Autoregressively generate a Lilylet document.
    Seeds with a BOS patch (+ optional metadata prompt), then samples patch by patch."""
    bos_patch = [BOS] * (PATCH_SIZE - 1) + [EOS]
    patches = [bos_patch]

    # optional textual prompt (metadata header lines) -> patches
    if prompt_text:
        for line in prompt_text.splitlines():
            ids = tokenizer.encode(line + '\n')
            for i in range(0, len(ids), PATCH_SIZE):
                chunk = ids[i:i + PATCH_SIZE]
                patches.append(chunk + [PAD] * (PATCH_SIZE - len(chunk)))

    out_text = patches_to_text(patches[1:])  # skip the bos control patch in display
    if verbose and out_text:
        print(out_text, end='')

    stream_started = False
    for step in range(max_patches):
        inp = torch.tensor([sum(patches, [])], device=device).reshape(1, -1, PATCH_SIZE)
        encoded = model.patch_level_decoder(inp)['last_hidden_state']  # [1, T, H]
        last = encoded[0, -1]

        patch_ids = generate_patch(last, temperature=temperature, top_k=top_k, top_p=top_p)
        if patch_stream and not stream_started and patch_to_text(patch_ids).startswith('[r:'):
            stream_started = True

        # EOS patch -> done
        if patch_ids[0] == BOS and patch_ids[1] == EOS:
            if verbose:
                print('\n[EOS patch -> stop]')
            break

        text = patch_to_text(patch_ids)
        out_text += text
        if verbose:
            print(text, end='')

        # mask tokens after the first EOS inside the patch to PAD before appending
        clean = list(patch_ids)
        seen_eos = False
        for j in range(len(clean)):
            if seen_eos:
                clean[j] = PAD
            if clean[j] == EOS:
                seen_eos = True
        patches.append(clean)
    else:
        if verbose:
            print('\n[max_patches reached]')
    return out_text

In [7]:
# Unconditional generation (start from BOS only).
torch.manual_seed(0)
text = generate(prompt_text='', max_patches=128, temperature=1.0, top_k=0, top_p=0.9)
print('\n\n===== generated text length:', len(text), 'chars =====')

[composer "Schubert, Franz"]


[genre "Romantic"]
[instrument "Keyboard"]


[r:0/45]\staff "1" \key e \major \time 3/4 \clef "treble" \tempo 4=

120 r4 \\
\staff "2" \clef "bass" r4 |


[r:1/44]\staff "1" \key e \major \time 3/4 r8 <g c 

...


\staff "2" e,2 e4 |
[r:21/24]\staff "1" \key e

 \major \time 3/4 r8 <e g b>( <g b e> <b 

g b> \\
\staff "2" e,2 e,4 |


[r:22/23]\staff "1" \key e \major \time 3/4 r8 <b e

 g>( <e g b> <b g b> \\
[max_patches reached]


===== generated text length: 2456 chars =====


In [8]:
# Conditional generation: seed with a metadata header and let the model continue.
torch.manual_seed(1)
prompt = '[title "Test Piece"]\n[composer "AI"]\n'
print('--- prompt ---')
print(prompt)
print('--- continuation ---')
text2 = generate(prompt_text=prompt, max_patches=128, temperature=0.9, top_k=20, top_p=0.95)
print('\n\n===== total length:', len(text2), 'chars =====')

--- prompt ---
[title "Test Piece"]
[composer "AI"]

--- continuation ---
[title "Test Piece"]
[composer "AI"]


[genre "Classical"]


[instrument "Art Song"]


[r:0/34]\key c \major \time 

6/8 \clef "treble" \tempo 4=92 r

...


\staff "2" e,4 c8 e4 c

8 |


[r:16/18]\staff "1" \key c

 \major \time 6/8 a'4 a8 

c4 b8 \\\
[max_patches reached]


===== total length: 2200 chars =====


In [9]:
# Greedy (deterministic) decode for a reproducible smoke check.
torch.manual_seed(0)
text3 = generate(prompt_text='', max_patches=64, temperature=1e-6, top_k=1, top_p=1.0, verbose=False)
print('greedy sample (first 400 chars):')
print(text3[:400])

greedy sample (first 400 chars):
[composer "Schubert, Franz"]
[genre "Romantic"]
[instrument "Chamber"]
[r:0/109]\key g \major \time 3/4 \clef "treble" \tempo 4=60 r2. \\\
\clef "treble" r4 <a f'>\pp( <g e'> \\\
\clef "alto" r4 <a f'>( <g e'> \\\
\clef "bass" r4 a\p( b \\\
\clef "bass" r4 f,\p( g |
[r:1/108]\key g \major \time 3/4 r2. \\\
r4 <d b'>( <d b'> \\\
r4 <d b'>( <d b'> \\\
r4 d( b \\\
r4 b( d \\\
r4 b( d \\\
r4 b( d \\\

